# Hybrid Quantum-Enhanced Linear Regression for Crop Yield Prediction

**Dataset:** Crop Yield with Soil and Weather

This notebook replaces the original Random Forest baseline with **Linear Regression** while preserving the quantum residual-correction pipeline:

1. Linear Regression baseline
2. QKRR (Quantum Kernel Ridge Regression) residual correction
3. VQR (Variational Quantum Regressor) final correction
4. Quantum kernel diagnostics
5. Residual analysis
6. 5-fold cross-validation and computational complexity

The central question is whether the quantum models can improve a classical Linear Regression model by learning its remaining residuals.


In [1]:
# ============================================================
# 1. IMPORTS AND SETUP
# ============================================================
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    median_absolute_error, max_error, explained_variance_score,
    mean_absolute_percentage_error
)

# Quantum imports
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms.regressors import VQR
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.state_fidelities import ComputeUncompute

np.random.seed(42)
from qiskit_algorithms.utils import algorithm_globals
algorithm_globals.random_seed = 42

print("All imports successful.")
print(f"Qiskit version: {__import__('qiskit').__version__}")


All imports successful.
Qiskit version: 2.5.0


In [3]:
# Quantum primitives
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
print("Quantum primitives imported successfully.")


Quantum primitives imported successfully.


In [6]:
# ============================================================
# 2. LOAD AND PREPROCESS DATA
# ============================================================
df = pd.read_csv(r"Crop Yiled with Soil and Weather.csv")

feature_cols = ["Fertilizer", "temp", "N", "P", "K"]
target_col = "yeild"

X = df[feature_cols].values
y = np.ravel(df[[target_col]].values)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_features = X.shape[1]
n_train = X_train_scaled.shape[0]
n_test = X_test_scaled.shape[0]

print(f"Training samples: {n_train}")
print(f"Test samples:     {n_test}")
print(f"Features:         {n_features}")
print(f"Feature ranges (scaled): [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")


Training samples: 2076
Test samples:     520
Features:         5
Feature ranges (scaled): [-1.874, 2.241]


## Part 1: Classical Linear Regression Baseline

Linear Regression is used as the classical baseline:

$$
\hat y = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p
$$

The quantum models are then asked to learn the residuals left by this baseline.


In [7]:
# ============================================================
# 3. CLASSICAL LINEAR REGRESSION BASELINE
# ============================================================
start = time.perf_counter()

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

lr_train_time = time.perf_counter() - start

start = time.perf_counter()
y_pred_lr = lr_model.predict(X_test_scaled)
lr_predict_time = time.perf_counter() - start

y_train_pred_lr = lr_model.predict(X_train_scaled)
lr_train_residuals = y_train - y_train_pred_lr

print("Linear Regression trained.")
print(f"Training time:    {lr_train_time:.6f}s")
print(f"Prediction time:  {lr_predict_time:.6f}s")
print(f"Training residual std: {np.std(lr_train_residuals):.6f}")

print("\nLinear Regression coefficients:")
for feature, coef in zip(feature_cols, lr_model.coef_):
    print(f"  {feature:12s}: {coef:.6f}")
print(f"  {'Intercept':12s}: {lr_model.intercept_:.6f}")


Linear Regression trained.
Training time:    0.114964s
Prediction time:  0.000813s
Training residual std: 0.706571

Linear Regression coefficients:
  Fertilizer  : -0.537947
  temp        : -0.690750
  N           : 0.832850
  P           : 0.422866
  K           : 0.503753
  Intercept   : 8.537630


In [8]:
# ============================================================
# 3b. EVALUATE CLASSICAL LINEAR REGRESSION
# ============================================================
def compute_metrics(y_true, y_pred, n, p):
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    smape_val = np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    ) * 100

    return {
        "R2": r2,
        "Adjusted R2": adj_r2,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Median AE": median_absolute_error(y_true, y_pred),
        "Max Error": max_error(y_true, y_pred),
        "Explained Var": explained_variance_score(y_true, y_pred),
        "MAPE (%)": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "SMAPE (%)": smape_val,
        "Mean Bias": np.mean(y_pred - y_true),
    }

lr_metrics = compute_metrics(y_test, y_pred_lr, n_test, n_features)

print("=== Classical Linear Regression Test Metrics ===")
for k, v in lr_metrics.items():
    print(f"  {k:15s}: {v:.6f}")


=== Classical Linear Regression Test Metrics ===
  R2             : 0.862648
  Adjusted R2    : 0.861312
  MAE            : 0.579096
  MSE            : 0.516205
  RMSE           : 0.718474
  Median AE      : 0.454576
  Max Error      : 2.011487
  Explained Var  : 0.863208
  MAPE (%)       : 6.888511
  SMAPE (%)      : 6.959818
  Mean Bias      : -0.045857


In [9]:
# ============================================================
# 3c. QUANTUM SUBSAMPLE
# ============================================================
# The quantum kernel is O(n^2) in simulator cost.
# We use 300 training and 150 test samples for the quantum stages.

X_train_q, _, y_train_q, _ = train_test_split(
    X_train_scaled, y_train,
    train_size=300,
    random_state=42
)

X_test_q, _, y_test_q, _ = train_test_split(
    X_test_scaled, y_test,
    train_size=150,
    random_state=42
)

n_train_q = X_train_q.shape[0]
n_test_q = X_test_q.shape[0]

print(f"Quantum train subset: {X_train_q.shape}")
print(f"Quantum test subset:  {X_test_q.shape}")

y_train_pred_lr_q = lr_model.predict(X_train_q)
y_pred_lr_q = lr_model.predict(X_test_q)

lr_train_residuals_q = y_train_q - y_train_pred_lr_q

print(
    f"\nLinear Regression residual std on quantum train subset: "
    f"{np.std(lr_train_residuals_q):.6f}"
)

print(f"\nQuantum train checksum: {X_train_q.sum():.6f}")
print(f"Quantum test  checksum: {X_test_q.sum():.6f}")


Quantum train subset: (300, 5)
Quantum test subset:  (150, 5)

Linear Regression residual std on quantum train subset: 0.727825

Quantum train checksum: -30.445774
Quantum test  checksum: 12.450402


## Part 2: Quantum Kernel Ridge Regression (QKRR) as Residual Corrector

QKRR learns the residuals left by Linear Regression.

$$
\hat y_{hybrid}
=
\hat y_{LR}
+
f_{QKRR}(x)
$$

The quantum kernel is constructed using a `ZZFeatureMap` and fidelity between quantum states.


In [10]:
# ============================================================
# 4. QUANTUM FEATURE MAP AND KERNEL
# ============================================================
n_qubits = n_features

feature_map = ZZFeatureMap(
    feature_dimension=n_qubits,
    reps=2,
    entanglement='full'
)

sampler = StatevectorSampler()
fidelity = ComputeUncompute(sampler=sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

print("Quantum feature map and kernel configured.")
print(f"Qubits: {feature_map.num_qubits}")
print(f"Circuit depth: {feature_map.decompose().depth()}")
print(f"Gate counts: {feature_map.decompose().count_ops()}")


Quantum feature map and kernel configured.
Qubits: 5
Circuit depth: 40
Gate counts: OrderedDict({'cx': 40, 'p': 30, 'h': 10})


In [11]:
# ============================================================
# 5. COMPUTE QUANTUM GRAM MATRICES
# ============================================================
n_evals = n_train_q * (n_train_q + 1) // 2 + n_test_q * n_train_q

print("Computing quantum kernel matrices...")
print(f"This requires {n_evals} circuit evaluations.")

start = time.perf_counter()

K_train_q = quantum_kernel.evaluate(x_vec=X_train_q)
K_test_q = quantum_kernel.evaluate(
    x_vec=X_test_q,
    y_vec=X_train_q
)

kernel_time = time.perf_counter() - start

print("\nQuantum kernel computation complete.")
print(f"Time: {kernel_time:.2f}s")
print(f"K_train_q shape: {K_train_q.shape}")
print(f"K_test_q shape:  {K_test_q.shape}")
print(f"K_train_q min/max: [{K_train_q.min():.4f}, {K_train_q.max():.4f}]")
print(f"K_train_q mean:    {K_train_q.mean():.4f}")

eigs = np.linalg.eigvalsh(K_train_q)
print(f"Min eigenvalue: {eigs.min():.6e}")
print(f"Is PSD: {eigs.min() >= -1e-10}")


Computing quantum kernel matrices...
This requires 90150 circuit evaluations.

Quantum kernel computation complete.
Time: 960.05s
K_train_q shape: (300, 300)
K_test_q shape:  (150, 300)
K_train_q min/max: [-0.0021, 1.0263]
K_train_q mean:    0.0489
Min eigenvalue: -1.291835e-15
Is PSD: True


In [12]:
# ============================================================
# 6. TRAIN QKRR ON LINEAR REGRESSION RESIDUALS
# ============================================================
lambda_reg = 1e-3

alpha_q = np.linalg.solve(
    K_train_q + lambda_reg * np.eye(n_train_q),
    lr_train_residuals_q
)

residual_correction_q = K_test_q @ alpha_q
y_pred_hybrid_q = y_pred_lr_q + residual_correction_q

hybrid_metrics_q = compute_metrics(
    y_test_q,
    y_pred_hybrid_q,
    n_test_q,
    n_features
)

print("=== Linear Regression + QKRR Test Metrics ===")
for k, v in hybrid_metrics_q.items():
    print(f"  {k:15s}: {v:.6f}")

print(f"\nCorrection magnitude (std): {np.std(residual_correction_q):.6f}")
print(
    f"Correction range: "
    f"[{residual_correction_q.min():.6f}, {residual_correction_q.max():.6f}]"
)


=== Linear Regression + QKRR Test Metrics ===
  R2             : -17.171780
  Adjusted R2    : -17.802745
  MAE            : 6.092717
  MSE            : 60.819946
  RMSE           : 7.798714
  Median AE      : 4.598002
  Max Error      : 20.107682
  Explained Var  : -15.457804
  MAPE (%)       : 74.276836
  SMAPE (%)      : 81.902654
  Mean Bias      : -2.395117

Correction magnitude (std): 7.469119
Correction range: [-20.809795, 15.160434]


In [13]:
# ============================================================
# 6b. FAIR COMPARISON: LR vs LR + QKRR
# ============================================================
lr_q_metrics = compute_metrics(
    y_test_q,
    y_pred_lr_q,
    n_test_q,
    n_features
)

comparison = pd.DataFrame({
    "Metric": list(lr_q_metrics.keys()),
    "Linear Regression (subset)": list(lr_q_metrics.values()),
    "LR + QKRR": list(hybrid_metrics_q.values()),
})

comparison["Improvement"] = (
    comparison["Linear Regression (subset)"] -
    comparison["LR + QKRR"]
)

comparison["Improvement %"] = (
    comparison["Improvement"] /
    comparison["Linear Regression (subset)"].abs()
) * 100

print("=== Fair Comparison on 150-Sample Quantum Test Subset ===")
print(comparison.to_string(index=False))


=== Fair Comparison on 150-Sample Quantum Test Subset ===
       Metric  Linear Regression (subset)  LR + QKRR  Improvement  Improvement %
           R2                    0.826835 -17.171780    17.998615    2176.807821
  Adjusted R2                    0.820823 -17.802745    18.623567    2268.890570
          MAE                    0.625815   6.092717    -5.466902    -873.565731
          MSE                    0.579573  60.819946   -60.240373  -10393.928623
         RMSE                    0.761297   7.798714    -7.037418    -924.398781
    Median AE                    0.509526   4.598002    -4.088476    -802.408310
    Max Error                    1.868593  20.107682   -18.239090    -976.086972
Explained Var                    0.829541 -15.457804    16.287345    1963.416835
     MAPE (%)                    7.311760  74.276836   -66.965077    -915.854461
    SMAPE (%)                    7.459901  81.902654   -74.442753    -997.905369
    Mean Bias                   -0.095159  -2.39511

## Part 3: Variational Quantum Regressor (VQR) as Final Corrector

VQR learns the residual remaining after Linear Regression and QKRR:

$$
r_{VQR}
=
y -
(\hat y_{LR}+f_{QKRR}(x))
$$

The final prediction is:

$$
\hat y_{final}
=
\hat y_{LR}
+
f_{QKRR}(x)
+
f_{VQR}(x)
$$


In [14]:
# ============================================================
# 7. VQR TARGETS
# ============================================================
qkrr_train_correction_q = K_train_q @ alpha_q

y_train_pred_hybrid_q = (
    y_train_pred_lr_q +
    qkrr_train_correction_q
)

vqr_targets_q = y_train_q - y_train_pred_hybrid_q

print("VQR target (remaining residual) stats:")
print(f"  Mean: {vqr_targets_q.mean():.6f}")
print(f"  Std:  {vqr_targets_q.std():.6f}")
print(f"  Min:  {vqr_targets_q.min():.6f}")
print(f"  Max:  {vqr_targets_q.max():.6f}")

if np.std(vqr_targets_q) < 1e-6:
    print("\nResiduals near-zero. Skipping VQR.")
    use_vqr = False
else:
    use_vqr = True
    print("\nResiduals significant. Proceeding with VQR.")


VQR target (remaining residual) stats:
  Mean: -0.000043
  Std:  0.052239
  Min:  -0.211879
  Max:  0.245453

Residuals significant. Proceeding with VQR.


In [16]:
# ============================================================
# 8. TRAIN VQR
# ============================================================
if 'vqr_targets_q' not in globals():
    raise RuntimeError(
        "Run the 'VQR TARGETS' cell first to create vqr_targets_q."
    )

if 'use_vqr' not in globals():
    use_vqr = np.std(vqr_targets_q) >= 1e-6

if use_vqr:
    ansatz = RealAmplitudes(
        num_qubits=n_qubits,
        reps=2,
        entanglement='full'
    )

    estimator = StatevectorEstimator()

    vqr = VQR(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=SPSA(maxiter=100),
        estimator=estimator,
        loss="squared_error"
    )

    print("Training VQR...")
    start = time.perf_counter()

    vqr.fit(
        X_train_q,
        np.asarray(vqr_targets_q).reshape(-1)
    )

    vqr_train_time = time.perf_counter() - start

    vqr_correction_q = np.asarray(
        vqr.predict(X_test_q)
    ).reshape(-1)

    y_pred_final_q = (
        np.asarray(y_pred_hybrid_q).reshape(-1) +
        vqr_correction_q
    )

    final_metrics_q = compute_metrics(
        y_test_q,
        y_pred_final_q,
        n_test_q,
        n_features
    )

    print(f"\nVQR training time: {vqr_train_time:.2f}s")
    print(f"VQR correction std: {np.std(vqr_correction_q):.6f}")

    print("\n=== LR + QKRR + VQR Test Metrics ===")
    for k, v in final_metrics_q.items():
        print(f"  {k:15s}: {v:.6f}")

else:
    print("Skipping VQR.")
    final_metrics_q = hybrid_metrics_q.copy()
    y_pred_final_q = y_pred_hybrid_q.copy()
    vqr_train_time = 0.0


Training VQR...

VQR training time: 294.93s
VQR correction std: 0.118592

=== LR + QKRR + VQR Test Metrics ===
  R2             : -17.110232
  Adjusted R2    : -17.739060
  MAE            : 6.081554
  MSE            : 60.613948
  RMSE           : 7.785496
  Median AE      : 4.563549
  Max Error      : 20.195837
  Explained Var  : -15.422781
  MAPE (%)       : 74.142012
  SMAPE (%)      : 81.742509
  Mean Bias      : -2.376511


In [17]:
# ============================================================
# 9. FINAL COMPARISON
# ============================================================
all_models = {
    "Linear Regression (subset)": lr_q_metrics,
    "LR + QKRR": hybrid_metrics_q,
}

if use_vqr:
    all_models["LR + QKRR + VQR"] = final_metrics_q

comparison_full = pd.DataFrame(all_models)
comparison_full.index.name = "Metric"

print("=" * 80)
print("FAIR COMPARISON (all evaluated on the same 150-sample test subset)")
print("=" * 80)
print(comparison_full.to_string())

print("\n" + "=" * 80)
print("KEY CHANGES VS LINEAR REGRESSION")
print("=" * 80)

for model_name in ["LR + QKRR", "LR + QKRR + VQR"]:
    if model_name in all_models:
        print(f"\n{model_name}:")

        for metric in ["R2", "RMSE", "MAE", "MAPE (%)"]:
            base = lr_q_metrics[metric]
            new = all_models[model_name][metric]

            if metric == "R2":
                pct = ((new - base) / abs(base)) * 100
                print(
                    f"  {metric:12s}: "
                    f"{base:.6f} -> {new:.6f} ({pct:+.2f}%)"
                )
            else:
                pct = ((base - new) / abs(base)) * 100
                print(
                    f"  {metric:12s}: "
                    f"{base:.6f} -> {new:.6f} "
                    f"({pct:+.2f}% reduction)"
                )


FAIR COMPARISON (all evaluated on the same 150-sample test subset)
               Linear Regression (subset)  LR + QKRR  LR + QKRR + VQR
Metric                                                               
R2                               0.826835 -17.171780       -17.110232
Adjusted R2                      0.820823 -17.802745       -17.739060
MAE                              0.625815   6.092717         6.081554
MSE                              0.579573  60.819946        60.613948
RMSE                             0.761297   7.798714         7.785496
Median AE                        0.509526   4.598002         4.563549
Max Error                        1.868593  20.107682        20.195837
Explained Var                    0.829541 -15.457804       -15.422781
MAPE (%)                         7.311760  74.276836        74.142012
SMAPE (%)                        7.459901  81.902654        81.742509
Mean Bias                       -0.095159  -2.395117        -2.376511

KEY CHANGES VS LINEAR 

## Part 4: Quantum-Specific Diagnostics

These diagnostics describe the quantum kernel and compare its representation with a classical RBF kernel.


In [18]:
# ============================================================
# 10. KERNEL ALIGNMENT (quantum vs classical RBF)
# ============================================================
from sklearn.metrics.pairwise import rbf_kernel

K_classical_q = rbf_kernel(X_train_q, gamma=1.0)

def kernel_alignment(K1, K2):
    return (
        np.sum(K1 * K2) /
        (np.linalg.norm(K1, 'fro') * np.linalg.norm(K2, 'fro'))
    )

alignment = kernel_alignment(
    K_train_q,
    K_classical_q
)

print(
    f"Kernel alignment (quantum vs classical RBF): "
    f"{alignment:.4f}"
)

if alignment > 0.9:
    print("  -> Very similar to RBF.")
elif alignment > 0.7:
    print("  -> Moderate similarity.")
else:
    print(
        "  -> Low similarity. "
        "Quantum kernel captures a substantially different representation."
    )


Kernel alignment (quantum vs classical RBF): 0.6155
  -> Low similarity. Quantum kernel captures a substantially different representation.


In [19]:
# ============================================================
# 11. LINEAR MODEL COEFFICIENTS + QUANTUM CIRCUIT DIAGNOSTICS
# ============================================================
coefficients = pd.Series(
    lr_model.coef_,
    index=feature_cols
).sort_values(
    key=np.abs,
    ascending=False
)

print("=== Linear Regression Coefficients ===")
print(coefficients.to_string())

print(f"\nIntercept: {lr_model.intercept_:.6f}")

print("\n=== Quantum Circuit Diagnostics ===")
print(f"Feature map qubits:      {feature_map.num_qubits}")
print(f"Feature map depth:       {feature_map.decompose().depth()}")
print(
    f"Feature map gate counts: "
    f"{feature_map.decompose().count_ops()}"
)

if use_vqr:
    print(f"VQR ansatz qubits:       {ansatz.num_qubits}")
    print(f"VQR ansatz depth:        {ansatz.decompose().depth()}")
    print(f"VQR parameters:          {ansatz.num_parameters}")


=== Linear Regression Coefficients ===
N             0.832850
temp         -0.690750
Fertilizer   -0.537947
K             0.503753
P             0.422866

Intercept: 8.537630

=== Quantum Circuit Diagnostics ===
Feature map qubits:      5
Feature map depth:       40
Feature map gate counts: OrderedDict({'cx': 40, 'p': 30, 'h': 10})
VQR ansatz qubits:       5
VQR ansatz depth:        15
VQR parameters:          15


In [20]:
# ============================================================
# 12. RESIDUAL ANALYSIS
# ============================================================
print("=== Residual Analysis (150-sample quantum test subset) ===")

models_and_preds = [
    ("Linear Regression (subset)", y_pred_lr_q),
    ("LR + QKRR", y_pred_hybrid_q),
]

if use_vqr:
    models_and_preds.append(
        ("LR + QKRR + VQR", y_pred_final_q)
    )

for name, y_pred in models_and_preds:
    res = y_test_q - y_pred

    print(f"\n{name}:")
    print(f"  Mean residual: {res.mean():.6f}")
    print(f"  Std residual:  {res.std():.6f}")
    print(f"  Skewness:      {pd.Series(res).skew():.4f}")
    print(f"  Kurtosis:      {pd.Series(res).kurtosis():.4f}")


=== Residual Analysis (150-sample quantum test subset) ===

Linear Regression (subset):
  Mean residual: 0.095159
  Std residual:  0.755326
  Skewness:      0.4226
  Kurtosis:      -0.3742

LR + QKRR:
  Mean residual: 2.395117
  Std residual:  7.421817
  Skewness:      0.0077
  Kurtosis:      -0.1542

LR + QKRR + VQR:
  Mean residual: 2.376511
  Std residual:  7.413916
  Skewness:      0.0049
  Kurtosis:      -0.1438


## Part 5: Cross-Validation and Complexity Analysis

Linear Regression is cheap enough to evaluate using standard 5-fold cross-validation.


In [21]:
# ============================================================
# 13. CROSS-VALIDATION (Linear Regression)
# ============================================================
print("Running 5-fold cross-validation for Linear Regression...")

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    LinearRegression(),
    X_train_scaled,
    y_train,
    cv=cv,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    return_train_score=True
)

print("\n5-Fold CV Results (Linear Regression):")
print(
    f"  R2   (test):  "
    f"{cv_results['test_r2'].mean():.4f} +/- "
    f"{cv_results['test_r2'].std():.4f}"
)
print(
    f"  MAE  (test):  "
    f"{-cv_results['test_mae'].mean():.4f} +/- "
    f"{cv_results['test_mae'].std():.4f}"
)
print(
    f"  RMSE (test):  "
    f"{-cv_results['test_rmse'].mean():.4f} +/- "
    f"{cv_results['test_rmse'].std():.4f}"
)
print(
    f"  R2   (train): "
    f"{cv_results['train_r2'].mean():.4f} +/- "
    f"{cv_results['train_r2'].std():.4f}"
)


Running 5-fold cross-validation for Linear Regression...

5-Fold CV Results (Linear Regression):
  R2   (test):  0.8659 +/- 0.0052
  MAE  (test):  0.5748 +/- 0.0144
  RMSE (test):  0.7089 +/- 0.0201
  R2   (train): 0.8672 +/- 0.0013


In [22]:
# ============================================================
# 14. COMPLEXITY SUMMARY
# ============================================================
print("=" * 70)
print("COMPUTATIONAL COMPLEXITY SUMMARY")
print("=" * 70)

print(f"""
CLASSICAL LINEAR REGRESSION:
  Training:  O(n * p^2 + p^3) for ordinary least squares
  Prediction: O(n * p)
  Space:     O(p^2)
  Actual training time: {lr_train_time:.6f}s
  Actual prediction time: {lr_predict_time:.6f}s

QUANTUM KERNEL RIDGE REGRESSION (QKRR):
  Kernel matrix (train): O(n_train^2 * C)
  Kernel matrix (test):  O(n_test * n_train * C)
  KRR solve:              O(n_train^3)
  Actual kernel time:     {kernel_time:.2f}s
  Qubits used:            {n_qubits}
  Circuit depth:          {feature_map.decompose().depth()}
""")

if use_vqr:
    print(f"""
VARIATIONAL QUANTUM REGRESSOR (VQR):
  Training:  O(iterations * batch * params * depth)
  Prediction: O(params * depth) per sample
  Actual training time: {vqr_train_time:.2f}s
  Parameters:           {ansatz.num_parameters}
  Ansatz depth:         {ansatz.decompose().depth()}
""")


COMPUTATIONAL COMPLEXITY SUMMARY

CLASSICAL LINEAR REGRESSION:
  Training:  O(n * p^2 + p^3) for ordinary least squares
  Prediction: O(n * p)
  Space:     O(p^2)
  Actual training time: 0.114964s
  Actual prediction time: 0.000813s

QUANTUM KERNEL RIDGE REGRESSION (QKRR):
  Kernel matrix (train): O(n_train^2 * C)
  Kernel matrix (test):  O(n_test * n_train * C)
  KRR solve:              O(n_train^3)
  Actual kernel time:     960.05s
  Qubits used:            5
  Circuit depth:          40


VARIATIONAL QUANTUM REGRESSOR (VQR):
  Training:  O(iterations * batch * params * depth)
  Prediction: O(params * depth) per sample
  Actual training time: 294.93s
  Parameters:           15
  Ansatz depth:         15



In [23]:
# ============================================================
# 15. SAVE QUANTUM KERNEL MATRICES
# ============================================================
np.save("K_train_q.npy", K_train_q)
np.save("K_test_q.npy", K_test_q)

print("Saved K_train_q.npy and K_test_q.npy")


Saved K_train_q.npy and K_test_q.npy


In [26]:
# ============================================================
# 16. QUANTUM CORRECTION DIAGNOSTICS
# ============================================================
print("QKRR correction statistics:")
print(f"Min: {residual_correction_q.min():.6f}")
print(f"Max: {residual_correction_q.max():.6f}")
print(f"Mean: {residual_correction_q.mean():.6f}")
print(f"Std: {residual_correction_q.std():.6f}")

print("\nLinear Regression residual standard deviation:")
print(f"{lr_train_residuals_q.std():.6f}")

print("\nCorrection/residual std ratio:")
print(f"{residual_correction_q.std() / max(lr_train_residuals_q.std(), 1e-12):.2f}x")


QKRR correction statistics:
Min: -20.809795
Max: 15.160434
Mean: -2.299957
Std: 7.469119

Linear Regression residual standard deviation:
0.727825

Correction/residual std ratio:
10.26x


## Summary

This notebook evaluates whether quantum residual correction can improve a **Linear Regression** baseline.

### Pipeline

$$
\text{Linear Regression}
\rightarrow
\text{QKRR residual correction}
\rightarrow
\text{VQR final correction}
$$

The key comparison is performed on the same 150-sample quantum test subset.

### Main research question

> Does the quantum correction reduce prediction error relative to Linear Regression alone?

The most important metrics are **R², RMSE, and MAE**. The residual and correction magnitudes are also reported to determine whether the quantum models are making useful corrections or overcorrecting the classical baseline.
